# Finance Advisor: No Memory vs. With Memory

A personal finance advisor agent built with the OpenAI Agents SDK. Below we compare two runs of the same two queries:
- **Task 1 & 2 (no memory):** each query is independent — no session attached.
- **Task 3 (with memory):** a `SQLiteSession` is attached so the agent remembers the expenses from query 1 when answering query 2.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner

load_dotenv()

In [ ]:
finance_advisor = Agent(
    name="Finance Advisor",
    model="gpt-5.4-mini",
    instructions=(
        "You are a personal finance advisor. Help the user track spending by category "
        "(e.g., groceries, transportation, dining, entertainment) and give practical, "
        "specific budget advice. When the user reports expenses, summarize the totals "
        "by category. When asked for advice, suggest concrete areas to cut back and "
        "explain your reasoning. Be concise and actionable."
    ),
)

## Tasks 1 & 2 — No Memory

Each `Runner.run` call is independent. The agent will not recall the expenses from query 1 when answering query 2.

In [ ]:
query1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."
result1 = await Runner.run(finance_advisor, query1)
print(result1.final_output)

In [ ]:
query2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"
result2 = await Runner.run(finance_advisor, query2)
print(result2.final_output)

## Task 3 — With Memory (SQLiteSession)

Now we attach a `SQLiteSession` to both runs. The session persists conversation history, so the second query has full context of the expenses reported in the first.

In [ ]:
from agents import SQLiteSession

session = SQLiteSession("finance_advisor_demo")

result1_mem = await Runner.run(finance_advisor, query1, session=session)
print("--- Query 1 (with memory) ---")
print(result1_mem.final_output)

In [ ]:
result2_mem = await Runner.run(finance_advisor, query2, session=session)
print("--- Query 2 (with memory) ---")
print(result2_mem.final_output)

### Side-by-side comparison

- **Without memory (Task 2):** the agent has no record of the $120 / $40 / $60 expenses, so the advice is generic.
- **With memory (Task 3):** the agent recalls the reported expenses ($220 total against a $250 budget) and gives concrete cuts grounded in those numbers.